In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
"""
Langevin Sampling — Step-by-Step Calculation Walkthrough
=========================================================
 
The update rule:
    x_{t+1} = x_t + eps * grad_log(x_t) + sqrt(2 * eps) * z_t
 
We'll compute EVERY term by hand (with code) so you can see
exactly what number goes where.
 
Example: Loaded die that favors face 6
    p(1)=0.05, p(2)=0.05, p(3)=0.10, p(4)=0.10, p(5)=0.20, p(6)=0.50
"""
import numpy as np
np.random.seed(42)

# ============================================================
# STEP 0: Define the target distribution (loaded die)
# ============================================================

face_probs = {1: 0.05, 2: 0.05, 3: 0.10, 4: 0.10, 5: 0.20, 6: 0.50}
print("=" * 65)
print("STEP 0: Target distribution (loaded die)")
print("=" * 65)
for face, prob in face_probs.items():
    bar = "#" * int(prob * 40)
    print(f"  Face {face}: p = {prob:.2f}  {bar}")
print()

STEP 0: Target distribution (loaded die)
  Face 1: p = 0.05  ##
  Face 2: p = 0.05  ##
  Face 3: p = 0.10  ####
  Face 4: p = 0.10  ####
  Face 5: p = 0.20  ########
  Face 6: p = 0.50  ####################



In [3]:
# ============================================================
# STEP 1: Build a smooth, differentiable p(x)
# ============================================================
# We can't take gradients of a discrete distribution.
# So we place a Gaussian bump at each face:
#
#   p(x) = sum over faces of:  weight_i * gaussian(x, mean=i, sigma=0.5)
#
# This creates a smooth curve with peaks at each face,
# and peak heights proportional to the face probabilities.


sigma = 0.5 # width of each gaussian bump
def p_conyinuous(x):
    """Smooth probability density built from weighted Gaussians."""
    total = 0.0
    for face, weight in face_probs.items():
        total += weight * np.exp(-0.5 * ((x - face) / sigma) ** 2)
    # Normalize by 1/(sigma * sqrt(2*pi)) for proper Gaussians,
    # but since we only need gradients of LOG p(x), the constant cancels out.
    return total

print("=" * 65)
print("STEP 1: Smooth p(x) at various positions")
print("=" * 65)
print("  We evaluate the smooth density at a few points:\n")
for x_val in [1.0, 2.0, 3.5, 5.0, 6.0]:
    print(f"  p({x_val:.1f}) = {p_continuous(x_val):.6f}")
print()
print("  Notice p(6.0) is the largest — the loaded die favors 6.\n")


STEP 1: Smooth p(x) at various positions
  We evaluate the smooth density at a few points:

  p(1.0) = 0.056800
  p(2.0) = 0.070334
  p(3.5) = 0.124085
  p(5.0) = 0.281235
  p(6.0) = 0.527101

  Notice p(6.0) is the largest — the loaded die favors 6.



In [5]:
# ============================================================
# STEP 2: Compute log p(x)
# ============================================================
def log_p(x):
    """Log of the smooth probability density."""
    return np.log(max(p_continuous(x), 1e-30)) # clamp to avoid log(0)


print("=" * 65)
print("STEP 2: log p(x) at the same positions")
print("=" * 65)
print("  Taking the log of p(x):\n")
for x_val in [1.0, 2.0, 3.5, 5.0, 6.0]:
    print(f"  log p({x_val:.1f}) = {log_p(x_val):.6f}")
print()
print("  Higher values = higher probability. log p(6.0) is the largest.\n")
 


STEP 2: log p(x) at the same positions
  Taking the log of p(x):

  log p(1.0) = -2.868213
  log p(2.0) = -2.654502
  log p(3.5) = -2.086785
  log p(5.0) = -1.268566
  log p(6.0) = -0.640364

  Higher values = higher probability. log p(6.0) is the largest.



In [8]:
# ============================================================
# STEP 3: Compute the gradient ∇log p(x) (numerical)
# ============================================================
# The gradient tells us: "which direction increases log p?"
#
# We approximate it with finite differences:
#   ∇log p(x) ≈ [log p(x + h) - log p(x - h)] / (2h)
 
h = 0.001  # small step for numerical derivative
 
def grad_log_p(x):
    """Numerical gradient of log p(x)."""
    return (log_p(x + h) - log_p(x - h)) / (2 * h)
 
print("=" * 65)
print("STEP 3: Gradient ∇log p(x) — which way to move?")
print("=" * 65)
print()
print("  Formula: ∇log p(x) ≈ [log p(x+h) - log p(x-h)] / (2h)")
print(f"  Using h = {h}\n")
 
for x_val in [1.0, 2.0, 3.5, 5.0, 6.0]:
    lp_plus  = log_p(x_val + h)
    lp_minus = log_p(x_val - h)
    grad     = (lp_plus - lp_minus) / (2 * h)
    print(f"  At x = {x_val:.1f}:")
    print(f"    log p({x_val + h:.3f}) = {lp_plus:.6f}")
    print(f"    log p({x_val - h:.3f}) = {lp_minus:.6f}")
    print(f"    gradient = ({lp_plus:.6f} - {lp_minus:.6f}) / {2*h} = {grad:+.4f}")
    if grad > 0.1:
        print(f"    → Positive gradient: PUSH RIGHT (toward higher faces)")
    elif grad < -0.1:
        print(f"    → Negative gradient: PUSH LEFT (toward lower faces)")
    else:
        print(f"    → Near zero: at or near a peak, little push")
    print()
 
 


STEP 3: Gradient ∇log p(x) — which way to move?

  Formula: ∇log p(x) ≈ [log p(x+h) - log p(x-h)] / (2h)
  Using h = 0.001

  At x = 1.0:
    log p(1.001) = -2.867733
    log p(0.999) = -2.868696
    gradient = (-2.867733 - -2.868696) / 0.002 = +0.4813
    → Positive gradient: PUSH RIGHT (toward higher faces)

  At x = 2.0:
    log p(2.001) = -2.654113
    log p(1.999) = -2.654891
    gradient = (-2.654113 - -2.654891) / 0.002 = +0.3887
    → Positive gradient: PUSH RIGHT (toward higher faces)

  At x = 3.5:
    log p(3.501) = -2.086704
    log p(3.499) = -2.086865
    gradient = (-2.086704 - -2.086865) / 0.002 = +0.0807
    → Near zero: at or near a peak, little push

  At x = 5.0:
    log p(5.001) = -1.267797
    log p(4.999) = -1.269335
    gradient = (-1.267797 - -1.269335) / 0.002 = +0.7690
    → Positive gradient: PUSH RIGHT (toward higher faces)

  At x = 6.0:
    log p(6.001) = -0.640571
    log p(5.999) = -0.640160
    gradient = (-0.640571 - -0.640160) / 0.002 = -0.2059
    →

In [9]:
# ============================================================
# STEP 4: One FULL Langevin update — every number shown
# ============================================================
print("=" * 65)
print("STEP 4: One full Langevin update, number by number")
print("=" * 65)
 
eps = 0.15  # step size
x_t = 2.0   # current position
 
print(f"""
  Update rule: x_{{t+1}} = x_t + eps * ∇log p(x_t) + sqrt(2*eps) * z_t
 
  Given:
    x_t   = {x_t}
    eps   = {eps}
""")
 
# Term 1: current position
print(f"  TERM 1 — Current position:")
print(f"    x_t = {x_t}")
print()
 
# Term 2: drift = eps * grad_log_p(x_t)
grad = grad_log_p(x_t)
drift = eps * grad
print(f"  TERM 2 — Drift (gradient push):")
print(f"    ∇log p({x_t}) = {grad:+.4f}")
print(f"    drift = eps * gradient = {eps} * {grad:+.4f} = {drift:+.6f}")
print(f"    Interpretation: pushes particle {abs(drift):.4f} to the {'right' if drift > 0 else 'left'}")
print()
 
# Term 3: noise = sqrt(2 * eps) * z
z_t = np.random.randn()  # one sample from N(0,1)
noise_scale = np.sqrt(2 * eps)
noise = noise_scale * z_t
print(f"  TERM 3 — Random noise:")
print(f"    z_t ~ N(0,1) = {z_t:+.4f}  (random draw)")
print(f"    noise_scale = sqrt(2 * eps) = sqrt(2 * {eps}) = {noise_scale:.4f}")
print(f"    noise = {noise_scale:.4f} * {z_t:+.4f} = {noise:+.6f}")
print()
 
# Combine
x_new = x_t + drift + noise
print(f"  COMBINING ALL THREE TERMS:")
print(f"    x_{{t+1}} = {x_t} + ({drift:+.6f}) + ({noise:+.6f})")
print(f"           = {x_new:.6f}")
print()
print(f"  Particle moved from {x_t:.1f} → {x_new:.4f}")
print(f"  Nearest die face: {round(x_new)}")
print()
 
 

STEP 4: One full Langevin update, number by number

  Update rule: x_{t+1} = x_t + eps * ∇log p(x_t) + sqrt(2*eps) * z_t
 
  Given:
    x_t   = 2.0
    eps   = 0.15

  TERM 1 — Current position:
    x_t = 2.0

  TERM 2 — Drift (gradient push):
    ∇log p(2.0) = +0.3887
    drift = eps * gradient = 0.15 * +0.3887 = +0.058298
    Interpretation: pushes particle 0.0583 to the right

  TERM 3 — Random noise:
    z_t ~ N(0,1) = +0.4967  (random draw)
    noise_scale = sqrt(2 * eps) = sqrt(2 * 0.15) = 0.5477
    noise = 0.5477 * +0.4967 = +0.272062

  COMBINING ALL THREE TERMS:
    x_{t+1} = 2.0 + (+0.058298) + (+0.272062)
           = 2.330360

  Particle moved from 2.0 → 2.3304
  Nearest die face: 2



In [10]:
# ============================================================
# STEP 5: Run 10 steps so you can see the pattern
# ============================================================
print("=" * 65)
print("STEP 5: Watch 10 consecutive steps")
print("=" * 65)
print()
print(f"  {'Step':>4}  {'x_t':>8}  {'gradient':>10}  {'drift':>10}  {'z_t':>8}  {'noise':>10}  {'x_new':>8}  {'face':>4}")
print(f"  {'----':>4}  {'--------':>8}  {'----------':>10}  {'----------':>10}  {'--------':>8}  {'----------':>10}  {'--------':>8}  {'----':>4}")
 
x = 2.0
for step in range(10):
    g = grad_log_p(x)
    d = eps * g
    z = np.random.randn()
    n = np.sqrt(2 * eps) * z
    x_next = x + d + n
    x_next = max(0.5, min(6.5, x_next))  # clamp to valid range
    face = int(np.round(x_next))
    face = max(1, min(6, face))
    print(f"  {step:>4}  {x:>8.4f}  {g:>+10.4f}  {d:>+10.6f}  {z:>+8.4f}  {n:>+10.6f}  {x_next:>8.4f}  {face:>4}")
    x = x_next
 
print()
print("  Watch how the gradient consistently pushes toward higher-")
print("  probability regions, while noise adds randomness.\n")
 


STEP 5: Watch 10 consecutive steps

  Step       x_t    gradient       drift       z_t       noise     x_new  face
  ----  --------  ----------  ----------  --------  ----------  --------  ----
     0    2.0000     +0.3887   +0.058298   -0.1383   -0.075730    1.9826     2
     1    1.9826     +0.3801   +0.057021   +0.6477   +0.354754    2.3943     2
     2    2.3943     +0.6554   +0.098312   +1.5230   +0.834198    3.3269     3
     3    3.3269     +0.0025   +0.000370   -0.2342   -0.128251    3.1990     3
     4    3.1990     +0.0317   +0.004751   -0.2341   -0.128242    3.0755     3
     5    3.0755     +0.1364   +0.020465   +1.5792   +0.864970    3.9609     4
     6    3.9609     +0.3727   +0.055907   +0.7674   +0.420341    4.4372     4
     7    4.4372     +0.7456   +0.111837   -0.4695   -0.257142    4.2919     4
     8    4.2919     +0.6205   +0.093073   +0.5426   +0.297172    4.6821     5
     9    4.6821     +0.8211   +0.123159   -0.4634   -0.253824    4.5514     5

  Watch how the

In [11]:
# ============================================================
# STEP 6: Full run — 5000 samples, compare to target
# ============================================================
print("=" * 65)
print("STEP 6: Full run with 5000 samples")
print("=" * 65)
 
N = 5000
burnin = 500
x = 3.5  # start in the middle
samples = []
 
for i in range(N + burnin):
    g = grad_log_p(x)
    d = eps * g
    z = np.random.randn()
    n = np.sqrt(2 * eps) * z
    x = x + d + n
    x = max(0.5, min(6.5, x))
    if i >= burnin:
        samples.append(x)
 
# Bin samples to nearest die face
counts = {f: 0 for f in range(1, 7)}
for s in samples:
    face = int(np.round(s))
    face = max(1, min(6, face))
    counts[face] += 1
 
print(f"\n  After {N} samples (burn-in {burnin}):\n")
print(f"  {'Face':>6}  {'Target':>8}  {'Sampled':>8}  {'Error':>8}  {'Histogram'}")
print(f"  {'----':>6}  {'------':>8}  {'-------':>8}  {'-----':>8}  {'---------'}")
 
for face in range(1, 7):
    target = face_probs[face]
    sampled = counts[face] / N
    error = abs(target - sampled)
    bar = "#" * int(sampled * 60)
    print(f"  {face:>6}  {target:>8.3f}  {sampled:>8.3f}  {error:>8.3f}  {bar}")
 
print()
print("  The sampled distribution closely matches the target!")
print("  More samples + smaller step size → even closer match.")
print()
 


STEP 6: Full run with 5000 samples

  After 5000 samples (burn-in 500):

    Face    Target   Sampled     Error  Histogram
    ----    ------   -------     -----  ---------
       1     0.050     0.062     0.012  ###
       2     0.050     0.074     0.024  ####
       3     0.100     0.079     0.021  ####
       4     0.100     0.114     0.014  ######
       5     0.200     0.256     0.056  ###############
       6     0.500     0.415     0.085  ########################

  The sampled distribution closely matches the target!
  More samples + smaller step size → even closer match.



In [12]:
# ============================================================
# STEP 7: Effect of step size (eps)
# ============================================================
print("=" * 65)
print("STEP 7: How step size (eps) affects quality")
print("=" * 65)
 
for test_eps in [0.01, 0.05, 0.15, 0.5, 1.0]:
    x = 3.5
    test_samples = []
    for i in range(N + burnin):
        g = grad_log_p(x)
        d = test_eps * g
        z = np.random.randn()
        n = np.sqrt(2 * test_eps) * z
        x = x + d + n
        x = max(0.5, min(6.5, x))
        if i >= burnin:
            test_samples.append(x)
 
    test_counts = {f: 0 for f in range(1, 7)}
    for s in test_samples:
        face = int(np.round(s))
        face = max(1, min(6, face))
        test_counts[face] += 1
 
    # KL divergence
    kl = 0
    for face in range(1, 7):
        p = face_probs[face]
        q = test_counts[face] / N
        if p > 0 and q > 0:
            kl += p * np.log(p / q)
 
    print(f"\n  eps = {test_eps:.2f}  →  KL divergence = {kl:.4f}  {'(too slow)' if test_eps < 0.05 else '(too wild)' if test_eps > 0.6 else '(good)'}")
    for face in range(1, 7):
        sampled = test_counts[face] / N
        bar = "#" * int(sampled * 40)
        print(f"    Face {face}: {sampled:.3f}  {bar}")
 
print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print("""
  The Langevin update has 3 terms:
 
  x_{t+1} = x_t + eps * ∇log p(x_t) + sqrt(2*eps) * z_t
             ───   ─────────────────   ─────────────────
              │           │                    │
              │           │                    └─ NOISE: random kick
              │           │                       keeps exploration alive
              │           │
              │           └─ DRIFT: gradient push
              │              moves toward high-probability regions
              │
              └─ POSITION: where you are now
 
  At equilibrium, drift (exploitation) and noise (exploration)
  balance perfectly, so the particle's histogram = target p(x).
""")


STEP 7: How step size (eps) affects quality

  eps = 0.01  →  KL divergence = 0.0881  (too slow)
    Face 1: 0.010  
    Face 2: 0.038  #
    Face 3: 0.113  ####
    Face 4: 0.160  ######
    Face 5: 0.295  ###########
    Face 6: 0.383  ###############

  eps = 0.05  →  KL divergence = 0.0525  (good)
    Face 1: 0.014  
    Face 2: 0.041  #
    Face 3: 0.063  ##
    Face 4: 0.107  ####
    Face 5: 0.285  ###########
    Face 6: 0.490  ###################

  eps = 0.15  →  KL divergence = 0.0237  (good)
    Face 1: 0.035  #
    Face 2: 0.056  ##
    Face 3: 0.086  ###
    Face 4: 0.127  #####
    Face 5: 0.271  ##########
    Face 6: 0.425  ################

  eps = 0.50  →  KL divergence = 0.0626  (good)
    Face 1: 0.057  ##
    Face 2: 0.070  ##
    Face 3: 0.114  ####
    Face 4: 0.176  #######
    Face 5: 0.244  #########
    Face 6: 0.338  #############

  eps = 1.00  →  KL divergence = 0.1242  (too wild)
    Face 1: 0.083  ###
    Face 2: 0.096  ###
    Face 3: 0.145  #####
    